In [11]:
import json
import os

import pandas as pd
from tqdm import tqdm

from utils.edinet_api import get_document, save_document, get_doc_name
from utils.datapath import edinet_codes_path, docs_metadata_path, documents_path
from utils.datetime import date_string_to_quarter
from utils.data import all_securities, jp_500, edinet_to_industry_map


In [12]:
# Tasks

# Verify doc type codes and include quarterly, semi-annually and yearly
# doc_type_codes = ["140", "160"]  # Quarterly and Semi-Annual Reports

# Get list of asset codes for Nikkei225 and industrials in particular - DONE!


# Get and store matching document list for Nikkei 225 companies for each year from 2016 onwards - DONE!

# Download each document - DONE!

# For each quarter, get latest submission date for the documents. Use that as the base date.
# Use that as a rebalance date
# In every base date, rank the 225 (or K) stocks in the basket using LLM

# For each stock in basket, get daily returns
# Perform L/S Equity

# MAIN BLOCKER:

# Get list of stocks - Done!
# Map to Edinet ID - Done!
# Understand how the mapping works - Done!
# ChatGPT API - Done!
# Time and motivation



In [13]:
df = pd.read_excel(edinet_codes_path)

with open(docs_metadata_path) as f:
    docs_metadata = json.load(f)

In [14]:
top_security_codes = list(set([code for quarter, codes in jp_500.items() for code in codes]))
top_securities = [security for security in all_securities if security["code"] in top_security_codes]
top_securities_edinet = [security["edinet_code"] for security in top_securities]
len(top_securities)

740

In [15]:
filtered_doc_metadata = [doc for doc in docs_metadata if doc["edinetCode"] in top_securities_edinet]
len(filtered_doc_metadata)

20175

In [17]:
# for doc in docs_metadata:
#     doc_id = doc['docID']
#     edinet_code = doc['edinetCode']
#     doc_type_code = doc['docTypeCode']
#     filer = doc['filerName']
#     save_name = f'{edinet_code}_{filer}_{doc_type_code}_{doc_id}.{FILE_EXT}'
#     output_path = os.path.join(documents_path, save_name)
#     doc_res = get_document(doc_id)
#     save_document(doc_res, output_path)

In [ ]:
# update save name to include date

for doc in tqdm(filtered_doc_metadata, desc="Processing documents"):
    industry = edinet_to_industry_map.get(doc['edinetCode'], "na")
    industry_path = os.path.join(documents_path, industry)
    if not os.path.isdir(industry_path):
        os.makedirs(industry_path)

    period_end_quater = date_string_to_quarter(doc["periodEnd"])
    quarter_path = os.path.join(industry_path, period_end_quater)
    if not os.path.isdir(quarter_path):
        os.makedirs(quarter_path)

    doc_id = doc['docID']    
    save_name = get_doc_name(doc)
    output_path = os.path.join(quarter_path, save_name)

    doc_res = get_document(doc_id)    
    save_document(doc_res, output_path)

Processing documents: 100%|██████████| 20175/20175 [3:18:42<00:00,  1.69it/s]  


In [ ]:
# Potentially merge industries?
# Analyse industry count for top securities